# Õppetund 10 - AI agendid tootmises

Selles õppetunnis õpite **tootmismustreid** AI agentidele, kasutades Microsoft Agent Frameworki (Python). Käsitleme:

- **Jälgitavus** — agentide suhtluste ajastuse ja logimise lisamine
- **Hindamine** — hindamisagendi kasutamine vastuste kvaliteedi hindamiseks
- **Kulude haldamine** — strateegiad tokenite optimeerimiseks ja mudeli valikuks

Stsenaarium on **reisiplaneerija**, kes aitab kasutajatel reise planeerida, millele on lisatud jälgimine ja hindamine.


## Seadistamine


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Tootmisalased kaalutlused

AI-agentide viimine prototüüpidest tootmisse nõuab hoolikat tähelepanu kolmele sambale:

1. **Jälgitavus** — Sul peab olema ülevaade sellest, mida agent teeb, kui kaua see aega võtab ja milliseid tööriistu ta kutsub. Ilma jälgimise ja logimiseta on tootmisprobleemide silumine peaaegu võimatu.

2. **Hindamine** — Automatiseeritud kvaliteedikontrollid tagavad, et agendi vastused jäävad aja jooksul täpsed, täielikud ja abistavad. Hindamisagent saab vastuseid määratletud kriteeriumide alusel hinnata.

3. **Kulude juhtimine** — Tokenite kasutamine mõjutab otseselt kulusid. Sellised strateegiad nagu prompti optimeerimine, mudeli valik ja vahemälu kasutamine aitavad hoida kulusid kontrolli all ilma kvaliteeti ohverdamata.


## Observeeritava agendi loomine

Määratleme reisimise tööriistad ja ümbritseme agendi kutsumise ajastusega, et saaksime latentsust jälgida. Tootmises integreeriksite OpenTelemetry'ga või mõne sarnase jälgimistaustsüsteemiga.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Hindamismustrid

Tavaline tootmismuster on kasutada teist agenti kui **hindajat**. Hindaja annab peamise agendi vastusele hinde vastavalt eelnevalt määratletud kriteeriumidele nagu täielikkus, täpsus ja abivalmidus.

See võimaldab:
- Automatiseeritud kvaliteedikontrolle enne, kui vastused kasutajateni jõuavad
- Regressiooni tuvastamist, kui promptid või mudelid muutuvad
- Agendi jõudluse pidevat jälgimist aja jooksul


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kulude juhtimise strateegiad

Kulude kontrollimine on tootmises kasutatavate tehisintellekti agentide jaoks kriitiline. Siin on peamised strateegiad:

| Strateegia | Kirjeldus |
|---|---|
| **Prompti optimeerimine** | Hoidke süsteemijuhised kokkuvõtlikud. Eemaldage liigne kontekst, et vähendada sisendtokenite arvu. |
| **Mudeli valik** | Kasutage lihtsate ülesannete (nt GPT-4o-mini) jaoks väiksemaid, odavamaid mudeleid nagu klassifitseerimine või väljavõte ning reserveerige suuremad mudelid keerukama mõtlemise jaoks. |
| **Vahemälu** | Salvestage tööriista tulemusi ja sagedasi päringuid, et vältida liigseid API-kõnesid. |
| **Tokeni eelarved** | Määrake `max_tokens` piirangud, et vältida ootamatult pikki vastuseid. |
| **Grupeerimine** | Grupeerige mitu kasutajapäringut ühe API-kõneks, kui võimalik. |

Praktikas töötab hästi kihiline lähenemine: suunake lihtsad päringud kiirele ja odavale mudelile ning eskaleerige ainult keerukad päringud võimekamale mudelile.


## Kokkuvõte

Selles õppetükis õppisite, kuidas:

1. **Lisada jälgitavust** agendi interaktsioonidele ajastuse ja logimise abil, rajades aluse jälitamisele ja monitooringule.
2. **Hinnata agendi vastuseid** automaatselt, kasutades hindamisagenti, kes skoorib täielikkuse, täpsuse ja kasulikkuse alusel.
3. **Haldada kulusid** läbi prompti optimeerimise, mudeli valiku, vahemälu kasutamise ja tokenite eelarvete.

Need tootmispraktikad aitavad tagada, et teie AI-agentid on mastaabis usaldusväärsed, mõõdetavad ja kulutõhusad.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Lahtiütlus:
See dokument on tõlgitud tehisintellekti tõlketeenuse Co-op Translator (https://github.com/Azure/co-op-translator) abil. Kuigi me püüame tagada täpsust, palun arvestage, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument algses keeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate väärarusaamade ega valede tõlgenduste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
